# 03_pc_<agreement>_<source>_to_<target> — core production pipeline template

Run-all-safe pipeline notebook for repeatable FabricOps production pipelines.

This base `03_pc` does not read DQ rules, quarantine records, fail fast on governance rules, or enforce sensitivity/classification/business decisions before governance has enhanced the metadata. It first creates clean pipeline evidence: selected agreement, notebook registration context, source and output profiles, catalogue evidence, deterministic transformation steps, runtime audit columns, target writes, read-back checks, and lineage.

`04_gov` is the place where governance users add or approve business context, schema expectations, data quality rules, sensitivity rules, classification rules, and enforcement decisions. After `04_gov` enriches the metadata, an enhanced production version of `03_pc` can load approved metadata and apply standard enforcement actions such as warn, split, quarantine, or stop.


## 1. Runtime setup

Load the shared FabricOps environment, path configuration, and metadata routing. Keep this active so the template remains plug-and-play in Microsoft Fabric.


In [ ]:
%run 00_env_config


## 2. Imports

Import only the helpers needed for the core pipeline flow. Governance enforcement helpers such as DQ enforcement, sensitivity enforcement, classification enforcement, and hard guardrail execution are intentionally deferred to the post-`04_gov` enhanced pipeline pattern.


In [ ]:
import json
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    setup_notebook,
    select_agreement,
    get_selected_agreement,
    current_notebook_active_registrations,
    read_lakehouse_csv,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
    profile_dataframe,
    add_runtime_audit_columns,
    build_lineage_records,
)


## 3. Pipeline parameters

Set these values for the source, target, catalogue evidence, lineage evidence, and pipeline identity. The default values use the starter-kit sample table and remain run-all safe once `00_env_config` has seeded sample data.


In [ ]:
USE_SAMPLE_DATA = True

ENV_NAME = ENV
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_KIND = "lakehouse"  # lakehouse | warehouse | csv | parquet
TARGET_KIND = "lakehouse"  # lakehouse | warehouse
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else "TODO_target_table"
SOURCE_FILE_PATH = "Files/sample/minimal_source.csv" if USE_SAMPLE_DATA else "Files/TODO/source_file.csv"
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "target_dataset"
TABLE_NAME = TARGET_TABLE
WRITE_MODE = "overwrite"
METADATA_WRITE_MODE = "append"
BUSINESS_KEYS = ["customer_id"]
PIPELINE_NAME = f"{SOURCE_TABLE}_to_{TARGET_TABLE}"
TOPIC = "production_pipeline"
RUN_ID = f"{PIPELINE_NAME}_{ENV_NAME}"

CATALOGUE_TABLE = "METADATA_DATA_CATALOGUE_COLUMN"
LINEAGE_TABLE = "METADATA_DATA_LINEAGE_TABLE"

if not USE_SAMPLE_DATA:
    setup_notebook(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product", "metadata"])

source_store = CONFIG.path_config.paths[ENV_NAME][SOURCE_LAYER]
target_store = CONFIG.path_config.paths[ENV_NAME][TARGET_LAYER]
metadata_store = CONFIG.path_config.paths[ENV_NAME]["metadata"]

runtime_context = {
    "run_id": RUN_ID,
    "environment": ENV_NAME,
    "dataset_name": DATASET_NAME,
    "source_kind": SOURCE_KIND,
    "target_kind": TARGET_KIND,
    "source_table": SOURCE_TABLE,
    "target_table": TARGET_TABLE,
    "pipeline_name": PIPELINE_NAME,
}


## 4. Data agreement selector and notebook registration

Select the data agreement and expose the notebook registration context for lineage and catalogue evidence. The selector registers this notebook as `03_pc` through the shared metadata route when the user clicks the registration button.

The current helper returns the selector widget, not a registration row. The lookup below reads active notebook registrations and uses `registration_id` when available. TODO: if the selector helper later exposes the created registration row directly, replace the lookup with that returned identifier.


In [ ]:
agreement_selector = select_agreement(
    CONFIG,
    ENV_NAME,
    spark_session=spark,
    register_notebook=True,
    notebook_type="03_pc",
    environment_name=ENV_NAME,
    dataset_name=DATASET_NAME,
    table_name=TABLE_NAME,
    topic=TOPIC,
    pipeline_name=PIPELINE_NAME,
)

selected_agreement = get_selected_agreement()
AGREEMENT_ID = str(selected_agreement.get("agreement_id") or agreement_selector.value or "")
AGREEMENT_CONTRACT_VERSION = str(selected_agreement.get("contract_version") or "")

active_notebook_registrations = current_notebook_active_registrations(
    spark,
    config=CONFIG,
    env=ENV_NAME,
    notebook_type="03_pc",
    environment_name=ENV_NAME,
)
matching_registration = next(
    (
        row for row in active_notebook_registrations
        if str(row.get("agreement_id") or "") == AGREEMENT_ID
        and str(row.get("agreement_contract_version") or "") == AGREEMENT_CONTRACT_VERSION
    ),
    None,
)
NOTEBOOK_REGISTRY_ID = (matching_registration or {}).get("registration_id")
NOTEBOOK_ID = (matching_registration or {}).get("notebook_id")

registration_context = {
    "agreement_id": AGREEMENT_ID,
    "agreement_contract_version": AGREEMENT_CONTRACT_VERSION,
    "notebook_registry_id": NOTEBOOK_REGISTRY_ID,
    "notebook_id": NOTEBOOK_ID,
}
display(registration_context)


## 5. Source read examples

Choose one active path with `SOURCE_KIND`. The lakehouse and warehouse examples use table helpers. The CSV and Parquet examples use Fabric lakehouse `Files/...` paths and are realistic for landing-zone or reference-file inputs.


In [ ]:
# Option A: Fabric lakehouse Delta table.
if SOURCE_KIND == "lakehouse":
    df_source = read_lakehouse_table(CONFIG, ENV_NAME, SOURCE_LAYER, SOURCE_TABLE, spark_session=spark)

# Option B: Fabric warehouse table.
elif SOURCE_KIND == "warehouse":
    df_source = read_warehouse_table(CONFIG, ENV_NAME, SOURCE_LAYER, "dbo", SOURCE_TABLE, spark_session=spark)

# Option C: CSV file from a lakehouse Files path.
elif SOURCE_KIND == "csv":
    df_source = read_lakehouse_csv(CONFIG, ENV_NAME, SOURCE_LAYER, SOURCE_FILE_PATH, spark_session=spark, header=True)

# Option D: Parquet file or folder from a lakehouse Files path.
elif SOURCE_KIND == "parquet":
    df_source = read_lakehouse_parquet(CONFIG, ENV_NAME, SOURCE_LAYER, SOURCE_FILE_PATH, spark_session=spark)

else:
    raise ValueError(f"Unsupported SOURCE_KIND: {SOURCE_KIND}")


## 6. Source profiling

Profile the selected source DataFrame and display the profile before transformation. This creates reusable evidence without applying governance enforcement.


In [ ]:
source_row_count = df_source.count()
source_profile = profile_dataframe(df_source, SOURCE_TABLE)
display(source_profile)


## 7. Write source catalogue evidence

Write source profile evidence into the reusable metadata data catalogue pattern. Do not create target-specific profile tables.


In [ ]:
def enrich_profile_for_catalogue(profile_df, *, evidence_role, table_name, layer, asset_kind, row_count):
    return (
        profile_df
        .withColumn("AGREEMENT_ID", F.lit(AGREEMENT_ID))
        .withColumn("AGREEMENT_CONTRACT_VERSION", F.lit(AGREEMENT_CONTRACT_VERSION))
        .withColumn("NOTEBOOK_REGISTRY_ID", F.lit(NOTEBOOK_REGISTRY_ID))
        .withColumn("NOTEBOOK_ID", F.lit(NOTEBOOK_ID))
        .withColumn("PROFILE_RUN_ID", F.lit(RUN_ID))
        .withColumn("ENVIRONMENT_NAME", F.lit(ENV_NAME))
        .withColumn("DATASET_NAME", F.lit(DATASET_NAME))
        .withColumn("PIPELINE_NAME", F.lit(PIPELINE_NAME))
        .withColumn("EVIDENCE_ROLE", F.lit(evidence_role))
        .withColumn("LAYER", F.lit(layer))
        .withColumn("ASSET_KIND", F.lit(asset_kind))
        .withColumn("PROFILED_TABLE_NAME", F.lit(table_name))
        .withColumn("PROFILED_ROW_COUNT", F.lit(row_count))
    )

source_catalogue_evidence = enrich_profile_for_catalogue(
    source_profile,
    evidence_role="source_profile",
    table_name=SOURCE_TABLE,
    layer=SOURCE_LAYER,
    asset_kind=SOURCE_KIND,
    row_count=source_row_count,
)

write_lakehouse_table(
    source_catalogue_evidence,
    CONFIG,
    ENV_NAME,
    "metadata",
    CATALOGUE_TABLE,
    mode=METADATA_WRITE_MODE,
)
source_catalogue_write_status = "written"


## 8. Pipeline-specific transformations

Replace this section with real transformation logic for your data product. Keep this base template deterministic and technical; do not mix governance enforcement into this section.


In [ ]:
df_transformed = df_source

if "status" in df_transformed.columns:
    df_transformed = df_transformed.withColumn("status", F.trim(F.lower(F.col("status"))))

if "email" in df_transformed.columns:
    df_transformed = df_transformed.withColumn("email", F.trim(F.lower(F.col("email"))))

if "amount" in df_transformed.columns:
    df_transformed = df_transformed.withColumn("amount", F.col("amount").cast("double"))
    df_transformed = df_transformed.withColumn("amount_band", F.when(F.col("amount") >= 100, F.lit("high")).otherwise(F.lit("standard")))


## 9. Runtime audit columns

`add_runtime_audit_columns(...)` adds only lightweight audit columns before the target write. These columns answer:

- which run produced the row: `_pipeline_run_id`
- which pipeline produced it: `_pipeline_name`
- which environment produced it: `_pipeline_environment`
- which source table it came from: `_source_table`
- when it was loaded: `_record_loaded_timestamp`
- which notebook produced it: `_notebook_name`
- who or what ran it: `_loaded_by`

Audit columns are always useful. Hash columns are only for deduplication, masked key comparison, slowly changing dimensions, or change detection. Datetime feature columns are analytics features, not audit fields. Bucket columns are only for advanced large-table layout or skew handling. For simple parallel data loading, use `repartition_by`. For physical Delta pruning, use `partition_by` with a natural column.


In [ ]:
df_output = add_runtime_audit_columns(
    df_transformed,
    run_id=RUN_ID,
    pipeline_name=PIPELINE_NAME,
    environment=ENV_NAME,
    source_table=SOURCE_TABLE,
)


## 10. Write output examples

Write to the selected target. Lakehouse and warehouse output examples are both shown, with the active path controlled by `TARGET_KIND`.


In [ ]:
if TARGET_KIND == "lakehouse":
    write_lakehouse_table(df_output, CONFIG, ENV_NAME, TARGET_LAYER, TARGET_TABLE, mode=WRITE_MODE)
elif TARGET_KIND == "warehouse":
    write_warehouse_table(df_output, CONFIG, ENV_NAME, TARGET_LAYER, "dbo", TARGET_TABLE, mode=WRITE_MODE)
else:
    raise ValueError(f"Unsupported TARGET_KIND: {TARGET_KIND}")


## Optional large table write pattern

For larger lakehouse tables, keep audit columns lightweight and tune the write itself. Simple parallel loading should use `repartition_by`, not framework bucket columns.

- `partition_by` controls the physical Delta table layout. Use natural pruning columns such as `event_date`, `ingestion_date`, `batch_date`, `source_file_date`, or `year_month`.
- `repartition_by` controls Spark dataframe parallelism before writing. Use column-based repartitioning for common medium/large writes, or an integer partition count for very large writes after testing.
- `repartition_by=2000` means Spark creates roughly 2000 dataframe partitions before writing. This is a tuning starting point, not a universal rule. Adjust it based on cluster size, target file size, skew, and write performance.


In [ ]:
# Optional example: large table with natural Delta partition columns and column-based Spark repartitioning.
# LARGE_TABLE_PARTITION_COLUMNS = ["event_date"]
# LARGE_TABLE_REPARTITION_BY = ["event_date"]
#
# write_lakehouse_table(
#     df_output,
#     CONFIG,
#     ENV_NAME,
#     TARGET_LAYER,
#     TARGET_TABLE,
#     mode=WRITE_MODE,
#     partition_by=LARGE_TABLE_PARTITION_COLUMNS,
#     repartition_by=LARGE_TABLE_REPARTITION_BY,
# )

# Optional example: very large table with an explicit dataframe partition count.
# LARGE_TABLE_PARTITION_COLUMNS = ["event_date"]
# LARGE_TABLE_REPARTITION_BY = 2000
#
# write_lakehouse_table(
#     df_output,
#     CONFIG,
#     ENV_NAME,
#     TARGET_LAYER,
#     TARGET_TABLE,
#     mode=WRITE_MODE,
#     partition_by=LARGE_TABLE_PARTITION_COLUMNS,
#     repartition_by=LARGE_TABLE_REPARTITION_BY,
# )


## 11. Read back output

Read the written target table back from the selected target so profiling and summary evidence reflect the published shape.


In [ ]:
if TARGET_KIND == "lakehouse":
    df_published = read_lakehouse_table(CONFIG, ENV_NAME, TARGET_LAYER, TARGET_TABLE, spark_session=spark)
else:
    df_published = read_warehouse_table(CONFIG, ENV_NAME, TARGET_LAYER, "dbo", TARGET_TABLE, spark_session=spark)


## 12. Output profiling

Profile the published output and display the profile.


In [ ]:
target_row_count = df_published.count()
output_profile = profile_dataframe(df_published, TARGET_TABLE)
display(output_profile)


## 13. Write output catalogue evidence

Write output profile evidence into the same reusable metadata catalogue pattern as the source evidence.


In [ ]:
output_catalogue_evidence = enrich_profile_for_catalogue(
    output_profile,
    evidence_role="output_profile",
    table_name=TARGET_TABLE,
    layer=TARGET_LAYER,
    asset_kind=TARGET_KIND,
    row_count=target_row_count,
)

write_lakehouse_table(
    output_catalogue_evidence,
    CONFIG,
    ENV_NAME,
    "metadata",
    CATALOGUE_TABLE,
    mode=METADATA_WRITE_MODE,
)
output_catalogue_write_status = "written"


## 14. Write lineage

Build source-to-target lineage records from consumed sources and produced targets. The records are tied to the selected agreement, run id, pipeline name, environment, and notebook registry id when available, then written to the metadata lineage table.


In [ ]:
lineage_records = build_lineage_records(
    dataset_name=DATASET_NAME,
    run_id=RUN_ID,
    source_tables=[SOURCE_TABLE],
    target_table=TARGET_TABLE,
    transformation_steps=[
        {
            "step": 1,
            "source": SOURCE_TABLE,
            "target": TARGET_TABLE,
            "operation": "deterministic transform + runtime audit columns",
        },
    ],
)

captured_at = datetime.now(timezone.utc).isoformat()
lineage_rows = []
for row in lineage_records:
    payload = dict(row)
    payload.update({"agreement_id": AGREEMENT_ID, "notebook_registry_id": NOTEBOOK_REGISTRY_ID})
    lineage_rows.append({
        "lineage_id": f"{RUN_ID}_{row.get('step')}",
        "run_id": RUN_ID,
        "agreement_id": AGREEMENT_ID,
        "agreement_contract_version": AGREEMENT_CONTRACT_VERSION,
        "environment_name": ENV_NAME,
        "dataset_name": DATASET_NAME,
        "pipeline_name": PIPELINE_NAME,
        "source_table": row.get("source") or SOURCE_TABLE,
        "target_table": row.get("target") or TARGET_TABLE,
        "notebook_registry_id": NOTEBOOK_REGISTRY_ID,
        "notebook_id": NOTEBOOK_ID,
        "lineage_level": "table",
        "transformation_type": "deterministic_pipeline",
        "transformation_summary": row.get("operation"),
        "captured_at": captured_at,
        "lineage_payload_json": json.dumps(payload, default=str),
    })

lineage_df = spark.createDataFrame(lineage_rows)
write_lakehouse_table(lineage_df, CONFIG, ENV_NAME, "metadata", LINEAGE_TABLE, mode=METADATA_WRITE_MODE)
lineage_write_status = "written"


## 15. Summary

Display a concise run summary for handover to the next workflow stage.


In [ ]:
run_summary = {
    "run_id": RUN_ID,
    "pipeline_name": PIPELINE_NAME,
    "environment": ENV_NAME,
    "source_table": SOURCE_TABLE,
    "source_row_count": source_row_count,
    "target_table": TARGET_TABLE,
    "target_row_count": target_row_count,
    "selected_agreement": AGREEMENT_ID,
    "agreement_contract_version": AGREEMENT_CONTRACT_VERSION,
    "notebook_registry_id": NOTEBOOK_REGISTRY_ID,
    "source_catalogue_write_status": source_catalogue_write_status,
    "output_catalogue_write_status": output_catalogue_write_status,
    "lineage_write_status": lineage_write_status,
    "catalogue_table": CATALOGUE_TABLE,
    "lineage_table": LINEAGE_TABLE,
}
display(run_summary)
